# PhantomNet ML System Walkthrough\n\nWelcome to the interactive walkthrough of the PhantomNet Machine Learning infrastructure. This notebook demonstrates the core components of our pipeline, from event classification to model management.

## 1. Environment Setup\nFirst, we ensure our environment is correctly configured.

In [ ]:
import os\nimport sys\nimport pandas as pd\nimport joblib\n\n# Ensure we're in the project root\nif os.path.basename(os.getcwd()) == 'notebooks':\n    os.chdir('../../')\n    \nprint(f\"Current Working Directory: {os.getcwd()}\")

## 2. Model Registry Overview\nWe use the `ModelRegistry` to manage versioned model artifacts.

In [ ]:
from ml.registry.model_registry import ModelRegistry\n\nregistry = ModelRegistry()\nlatest_version = registry.get_latest_version()\nmodel_meta = registry.get_model(latest_version)\n\nprint(f\"Latest Model Version: {latest_version}\")\nprint(f\"Model Status: {model_meta['status']}\")\nprint(f\"Metrics: {model_meta['metrics']}\")

## 3. Data Inspection\nLet's look at the enhanced dataset used for evaluation.

In [ ]:
eval_data_path = \"backend/ml/datasets/labeled_events_v2_enhanced.csv\"\ndf = pd.read_csv(eval_data_path)\ndf.head()

## 4. Manual Inference Test\nTesting the production model against a sample event.

In [ ]:
model = joblib.load(model_meta['path'])\nsample_event = df.drop('label', axis=1).iloc[0:1]\nprediction = model.predict(sample_event)\nconfidence = model.predict_proba(sample_event)[0][1] if hasattr(model, 'predict_proba') else \"N/A\"\n\nprint(f\"Classification: {'Malicious' if prediction[0] == 1 else 'Benign'}\")\nprint(f\"Confidence Score: {confidence}\")

## 5. Performance History\nVisualizing how our models have improved over time.

In [ ]:
import json\nhistory_path = \"ml/evaluation/evaluation_history.jsonl\"\n\nhistory = []\nwith open(history_path, \"r\") as f:\n    for line in f:\n        history.append(json.loads(line))\n\nhistory_df = pd.DataFrame(history)\nhistory_df[['version', 'accuracy', 'f1_score', 'precision', 'recall']]